# API  Nano Banana con Gemini 🍌

| **Modelo API**                                   | **Precio por imagen (USD)**      | **Costo por 1M tokens de input (texto)** | **Costo por 1M tokens de output (imagen)** |
| ------------------------------------------------ | -------------------------------- | ---------------------------------------- | ------------------------------------------ |
| **gemini-3.1-flash-image-preview** | ~$0.045 per 0.5K image* $0.067 per 1K image*, $0.101 per 2K image*, and $0.151 per 4K image* | ---------------------------------------- | ------------------------------------------ |
| **Nano Banana (gemini-2.5-flash-image)**         | ~$0.039 por imagen               | $0.15 / 1M tokens                        | $30 / 1M output tokens*                    
| **Nano Banana Pro (gemini-3-pro-image-preview)** | ~$0.134 por 1K-2K, ~$0.24 por 4K | $0.15 / 1M tokens                        | $30 / 1M output tokens*                    |


## Objetivo
- Configurar cliente Gemini con API Key local.
- Generar y editar imágenes con `gemini-2.5-flash-image`.
- Generar imágenes con `gemini-3-pro-image-preview`.
- Usar un chat multimodal con herramientas de búsqueda.



## 1. Instalación de dependencias

Ejecuta esta celda una sola vez por entorno.

Nota: la línea está comentada para evitar reinstalar en cada ejecución. Descoméntala si estás en un entorno nuevo.


In [ ]:
# !pip install -q -U google-genai pillow

## 2. Configuración del cliente e imports

En esta sección:
- Se cargan librerías necesarias.
- Se valida la existencia del archivo de credenciales.
- Se inicializa un cliente único reutilizable en todo el notebook.

Formato esperado de `API_KEY_GEMINI.json`:
```json
{
  "API_KEY_PAY": "tu_clave_aqui"
}
```


In [1]:
import json
from pathlib import Path
from typing import Optional

from google import genai
from google.genai import types
from IPython.display import Markdown, display
from PIL import Image

# Archivo local con la credencial de Gemini.
API_KEY_PATH = Path("API_KEY_GEMINI.json")

# Validación temprana para evitar errores posteriores difíciles de depurar.
if not API_KEY_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró {API_KEY_PATH}. Crea el archivo con la clave API antes de continuar."
    )

with API_KEY_PATH.open("r", encoding="utf-8") as f:
    api_key_data = json.load(f)

# Nombre de variable explícito para mejorar legibilidad.
GEMINI_API_KEY = api_key_data["API_KEY_PAY"]

# Cliente único reutilizado en todo el notebook.
client = genai.Client(api_key=GEMINI_API_KEY)
print("Cliente Gemini configurado correctamente.")


## 3. Ejemplo de texto (Q&A rápido)

Se envía una pregunta simple al modelo `gemini-2.5-flash` y se muestra la respuesta en Markdown para mejorar legibilidad.


In [ ]:
text_prompt = "¿Qué es la inteligencia artificial?"

response = client.models.generate_content(
    model="gemini-2.5-flash", # no consume dinero, pero no tiene capacidades de imagen.
    contents=text_prompt,
)

# Mostramos la respuesta como Markdown para mejor lectura en notebook.
display(Markdown(response.text))


## 4. Utilidad para procesar respuestas multimodales

La función `save_response_parts` centraliza el manejo de respuestas:
- Imprime bloques de texto.
- Guarda la primera imagen recibida en disco (si se define ruta de salida).

Esto evita duplicación de código en cada ejemplo.


In [2]:
def save_response_parts(response, output_image_path: Optional[str] = None) -> None:
    """
    Procesa la respuesta del modelo Gemini.

    - Imprime cualquier texto generado.
    - Muestra imágenes generadas en notebook.
    - Opcionalmente guarda la imagen en disco si se especifica una ruta.
    """

    # Verifica que la respuesta tenga candidatos (puede no tenerlos si hubo error o bloqueo)
    if not hasattr(response, "candidates"):
        print("No candidates found.")
        return

    # Recorre cada posible candidato generado por el modelo
    # (puede haber más de uno si usaste candidate_count > 1)
    for candidate in response.candidates:

        # Asegura que el candidato tenga contenido válido
        if not hasattr(candidate, "content"):
            continue

        # Cada candidato puede tener múltiples partes (texto, imagen, tool call, etc.)
        for part in candidate.content.parts:
            
            # Si la parte contiene texto, lo imprime
            if getattr(part, "text", None):
                print(part.text)

            # Si la parte contiene datos binarios (ej: imagen)
            elif getattr(part, "inline_data", None):
                
                # Convierte automáticamente el base64 en objeto PIL.Image
                image = part.as_image()
                
                # Muestra la imagen en notebook (Jupyter/Colab)
                #display(image)

                # Si se proporcionó ruta de salida, guarda la imagen en disco
                if output_image_path:
                    image.save(output_image_path)
                    print(f"Imagen guardada en: {output_image_path}")

## 5. Generación de imagen desde texto

Entrada: prompt textual.
Salida esperada: archivo `generated_image.png`.


In [4]:
image_prompt = "Create a picture of a banana dish in a fancy restaurant with a jungle theme"

response = client.models.generate_content(
    model="gemini-2.5-flash-image",
    contents=[image_prompt],
)

save_response_parts(response, output_image_path="generated_image.png")


### Vista previa del resultado

La siguiente celda abre y muestra la imagen recién generada (`generated_image.png`).


In [5]:
# Carga la imagen generada y muéstrala en el notebook.
generated_image_1 = Image.open("generated_image.png")
display(generated_image_1)

## 6. Edición de imagen usando una foto local

Entrada:
- Prompt de edición.
- Imagen local (`cat_image.jpg` y `dog_image.jpg`).

Salida esperada: archivo `generated_image_from_cat.png`.

Si no existe la imagen local, la celda muestra un mensaje y no falla.


In [ ]:
local_cat_image_path = Path("cat_image.jpg")  
local_dog_image_path = Path("dog_image.jpg")  

edit_prompt = (
    "Create a picture of my cat and my dog sitting together at a table in a "
    "The cat is eating a dish while the dog looks curious and excited. "
    "Maintain the original appearance and features of both pets."
)

if not local_cat_image_path.exists() or not local_dog_image_path.exists():
    print(
        "No se encontró una de las imágenes. "
        "Actualiza las rutas para probar la edición de imagen."
    )
else:
    cat_image = Image.open(local_cat_image_path)
    dog_image = Image.open(local_dog_image_path)

    response = client.models.generate_content(
        model="gemini-2.5-flash-image",
        contents=[
            edit_prompt,
            cat_image,
            dog_image,
        ],
    )

    save_response_parts(
        response,
        output_image_path="generated_image_cat_and_dog.png"
    )

### Vista previa de la imagen editada

La siguiente celda abre `generated_image_cat_and_dog.png` para validar el resultado de edición.


In [8]:
# Carga la imagen generada y muéstrala en el notebook.
generated_image_2 = Image.open("generated_image_cat_and_dog.png")
display(generated_image_2)

## 7. Chat multimodal: crear infografía y luego traducirla

Flujo en dos pasos dentro de la misma conversación (`chat`):
1. Crear una infografía educativa (texto + imagen).
2. Reutilizar contexto para traducir la misma infografía a español.

Archivos de salida:
- `ai_pipeline.png`
- `ai_pipeline_spanish.png`


In [9]:
chat = client.chats.create(
    model="gemini-3-pro-image-preview",
    config=types.GenerateContentConfig(
        response_modalities=["TEXT", "IMAGE"],
        tools=[{"google_search": {}}],
    ),
)

infographic_prompt = (
    "Create a modern, visually clean infographic that explains Artificial Intelligence "
    "as a layered learning pipeline for graduate students. Show the key components: "
    "data acquisition, preprocessing, feature engineering, model selection, training, "
    "validation, optimization, and deployment. Include visual metaphors such as a "
    "neural network diagram, a data pipeline flow, and performance metrics (RMSE, F1-score, "
    "AUC). The style should resemble a professional academic conference poster with "
    "minimalist design, clear diagrams, and concise technical explanations."
)

response = chat.send_message(infographic_prompt)
save_response_parts(response, output_image_path="ai_pipeline.png")


### Vista previa de la primera infografía

La siguiente celda muestra `photosynthesis.png`.


In [10]:
# Carga la imagen generada y muéstrala en el notebook.
generated_image_2 = Image.open("ai_pipeline.png")
display(generated_image_2)

In [11]:
# Segunda interacción del mismo chat: mantenemos contexto y pedimos traducción.
translation_prompt = (
    "Update this infographic to be in Spanish. "
    "Do not change any other elements of the image."
)

aspect_ratio = "16:9"  # Opciones: "1:1", "2:3", "3:2", "3:4", "4:3", "4:5", "5:4", "9:16", "16:9", "21:9"
resolution = "2K"      # Opciones: "1K", "2K", "4K"

response = chat.send_message(
    translation_prompt,
    config=types.GenerateContentConfig(
        image_config=types.ImageConfig(
            aspect_ratio=aspect_ratio,
            image_size=resolution,
        ),
    ),
)

save_response_parts(response, output_image_path="ai_pipeline_spanish.png")


### Vista previa de la infografía traducida

La siguiente celda muestra `ai_pipeline_spanish.png`.


In [12]:
# Carga la imagen generada y muéstrala en el notebook.
generated_image_3 = Image.open("ai_pipeline_spanish.png")
display(generated_image_3)

## 8. Ejemplo adicional: pronóstico del clima en gráfico

Se solicita una visualización meteorológica para 5 días usando modalidad multimodal.

Salida esperada: archivo `weather.png`.


In [28]:
weather_prompt = (
    "Visualize the current weather forecast for the next 5 days in Medellin, Colombia"
    "as a clean, modern weather chart. Add a visual on what I should wear each day."
)
weather_aspect_ratio = "16:9"

response = client.models.generate_content(
    model="gemini-3-pro-image-preview",
    contents=weather_prompt,
    config=types.GenerateContentConfig(
        response_modalities=["TEXT", "IMAGE"],
        image_config=types.ImageConfig(aspect_ratio=weather_aspect_ratio),
        tools=[{"google_search": {}}],
    ),
)

save_response_parts(response, output_image_path="weather.png")


### Vista previa del gráfico de clima

La siguiente celda abre y muestra `weather.png`.


In [29]:
generated_image_4 = Image.open("weather.png")
display(generated_image_4)

### Actividades

**1. Visualización de datos con búsqueda en tiempo real**
Crea un prompt que utilice `google_search` para obtener datos actuales sobre un tema cuantificable (precio del dólar, petróleo Brent, calidad del aire, resultados deportivos, etc.) y genere una visualización gráfica clara y profesional. El prompt debe especificar estilo visual (colores, tipo de gráfico, títulos, fuentes, anotaciones).



**2. Flujo texto → imagen (pipeline generativo en cascada)**
Diseña un flujo de dos pasos: primero usa `gemini-2.5-flash` (solo texto) para generar una descripción técnica y detallada de una escena imaginaria (incluye iluminación, materiales, composición, lente, estilo artístico y atmósfera). Luego utiliza exactamente esa descripción como prompt para generar la imagen con `gemini-2.5-flash-image`. Finalmente, compara el resultado con una imagen que habrías generado usando tu propio prompt manual y analiza diferencias en fidelidad visual, nivel de detalle y control semántico.



**3. Chat multimodal con memoria de ediciones progresivas**
Construye una sesión de chat donde realices mínimo cuatro rondas de edición acumulativa sobre una misma imagen (por ejemplo: crear personaje → agregar fondo → modificar iluminación → añadir texto → introducir cambio potencialmente conflictivo). Cada turno debe añadir un cambio nuevo sin repetir instrucciones anteriores. Al finalizar, pide al modelo que describa técnicamente la imagen final y analiza si mantiene coherencia visual, identidad del personaje y consistencia en iluminación y estilo a lo largo de los turnos.
